# Cấp độ 2 — GRKT (KDD 2024) trên Kaggle GPU

Notebook **clone repo** [Knowledge-Tracing](https://github.com/linhnnh688/Knowledge-Tracing) rồi gọi hàm từ package `level2/`. Toàn bộ logic thực nghiệm nằm trong package; notebook chỉ điều phối — cùng kiến trúc với `kaggle_level1.ipynb`.

Mô hình chính: **GRKT — Knowledge Tracing với truy xuất trên đồ thị quan hệ** (Cui et al., KDD 2024), mã nguồn vendored trong thư mục `GRKT/`.

Phạm vi notebook = đúng yêu cầu Cấp độ 2: chạy GRKT trên **≥ 2 tập dữ liệu công khai** với cùng quy trình chia dữ liệu và cùng chỉ số, trực quan hoá mastery theo khái niệm/thời gian, đo ≥ 2 tiêu chí hợp lý của quỹ đạo. Ablation cấu trúc đồ thị và đa seed thuộc Cấp độ 1 (đã có kết quả) nên mặc định TẮT ở đây; calibration là phân tích nâng cao thứ hai, đề bài chỉ cần chọn **một** — đã chọn quỹ đạo mastery.

| Yêu cầu đề bài (Cấp độ 2) | Triển khai trong notebook |
|---|---|
| Mô hình graph-aware từ 2024+ | **GRKT**: 3 cơ chế *retrieval / strengthening / learning-forgetting* trên `rel_map` + `pre_map` |
| ≥ 2 tập dữ liệu công khai, cùng quy trình | `assist2009` (CSV chuẩn hoá của Cấp độ 1) và `junyi` (Kaggle) — cùng split theo người học, cùng chỉ số |
| Chống rò rỉ | dùng lại **đúng split seed 42 của Cấp độ 1**; `rel_map`/`pre_map` chỉ đếm từ tập train; in `leak_report.txt`; chuỗi GRKT được **đối chiếu từng người học** với bundle Cấp độ 1 |
| Trực quan hoá mastery theo thời gian | `mastery_<graph>.csv` = bảng dài m(u,c,t), nhiệt độ khái niệm × thời điểm, đường cong mastery |
| ≥ 2 tiêu chí hợp lý của quỹ đạo | **hướng lan truyền** (`consistency_spread`), **tính cục bộ** (`locality`), **suy giảm theo thời gian** (`forget_slope`) |
| So sánh cùng quy trình | bảng DKT / GKT-Transition / GKT-Zero / GRKT trên cùng split (đường nền **nạp checkpoint Cấp độ 1**, không train lại) |
| Ablation cấu trúc đồ thị | thuộc **Cấp độ 1** (đã có kết quả); GRKT hỗ trợ `train`/`dense`/`zero`/`shuffled` khi bật `RUN_GRAPH_ABLATIONS=True` |
| Phân tích nâng cao (đề bài chỉ cần **một**) | đã chọn **quỹ đạo mastery + tiêu chí hợp lý**; ECE/Brier có sẵn trong mã, bật `RUN_CALIBRATION=True` |
| Đa seed | DKT/GKT đã có Mean ± Std 3 seed ở Cấp độ 1 trên cùng split; `RUN_MULTI_SEED=True` nếu muốn thêm cho GRKT |

**Trước khi chạy**: tìm dataset **Junyi Academy** trong Kaggle Datasets, bấm **Agree/Subscribe** rồi `Add input` vào notebook. Package sẽ tự dò file trong `/kaggle/input` (không cần gõ đường dẫn). Nếu chưa attach, cell chuẩn hoá sẽ báo rõ là thiếu gì.

## 0. Clone repo và cấu hình môi trường

In [ ]:
import os
import sys
import shutil
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/linhnnh688/Knowledge-Tracing.git'
KAGGLE_ROOT = Path('/kaggle/working')
IS_KAGGLE = KAGGLE_ROOT.exists()

if IS_KAGGLE:
    REPO = KAGGLE_ROOT / 'Knowledge-Tracing'
    if not (REPO / '.git').exists():
        if REPO.exists():
            shutil.rmtree(REPO, ignore_errors=True)
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(REPO)])
    else:
        try:
            subprocess.check_call(['git', '-C', str(REPO), 'fetch', 'origin', 'main', '--depth', '1'])
            subprocess.check_call(['git', '-C', str(REPO), 'reset', '--hard', 'origin/main'])
        except Exception as e:
            print(f'Git sync error ({e}), re-cloning...')
            shutil.rmtree(REPO, ignore_errors=True)
            subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(REPO)])
else:
    REPO = Path.cwd() if (Path.cwd() / 'level2').exists() else Path.cwd() / 'Knowledge-Tracing'

os.chdir(REPO)

# REPO và REPO/GKT phải nằm trong sys.path: level1/ nạp pipeline của GKT theo kiểu module-top-level
for p in [REPO, REPO / 'GKT']:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

# Git ở trên vừa thay code trên đĩa. Nếu kernel này ĐÃ import level1/level2/GRKT/GKT trước đó
# thì Python vẫn giữ module cũ trong sys.modules, và code mới sẽ không có tác dụng cho tới khi
# restart kernel — dấu hiệu hay gặp là "ImportError: cannot import name 'X' from 'level2...'"
# ngay sau khi cell clone chạy thành công. Xoá cache ở đây để chạy lại notebook từ giữa cũng
# ăn được code mới mà không phải Restart kernel.
_stale = [m for m in list(sys.modules) if m.split(".")[0] in ("level1", "level2", "GRKT", "GKT")]
for m in _stale:
    sys.modules.pop(m, None)
if _stale:
    print(f"Đã bỏ {len(_stale)} module cũ khỏi sys.modules (import lại sẽ lấy code vừa clone).")
else:
    print("sys.modules chưa có module dự án nào — không phải bỏ gì.")

try:
    _head = subprocess.check_output(["git", "-C", str(REPO), "log", "--oneline", "-1"], text=True).strip()
except Exception as _e:
    _head = f"(không đọc được git: {_e})"
print("HEAD đang dùng:", _head)

print('Repo directory:', REPO.resolve())
print('Cấp độ 2 package:', (REPO / 'level2').resolve(), '| tồn tại:', (REPO / 'level2').is_dir())
print('GRKT vendored   :', (REPO / 'GRKT').resolve(), '| tồn tại:', (REPO / 'GRKT').is_dir())

In [ ]:
# Kaggle GPU image đã có sẵn PyTorch bản chạy được GPU + pandas/numpy/matplotlib/sklearn/tqdm.
# TUYỆT ĐỐI không `pip install torch` (sẽ tải bản CPU ~800MB và làm hỏng môi trường), và cũng
# không cố định phiên bản pandas/numpy: image hiện chạy Python 3.13 + pandas 3.x, code trong
# level1/level2 đã viết tương thích cả pandas 2 lẫn 3 (vẽ hình bằng matplotlib thuần).
import importlib.util
import subprocess
import sys

# tên module -> tên gói pip, chỉ cài những thứ thật sự thiếu
OPTIONAL = {"pandas": "pandas", "scipy": "scipy", "sklearn": "scikit-learn",
            "matplotlib": "matplotlib", "tqdm": "tqdm"}
missing = [pkg for mod, pkg in OPTIONAL.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Cài bổ sung (chỉ các gói chưa có):", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("Mọi thư viện đã có sẵn trong image — không cài gì thêm.")

import pandas as _pd, numpy as _np, matplotlib as _mpl
print("python", sys.version.split()[0], "| numpy", _np.__version__, "| pandas", _pd.__version__,
      "| matplotlib", _mpl.__version__)


### 0.2 Cấu hình thực nghiệm

* `QUICK_RUN = True` — dùng tập `assist2009_smoke` (15 người học, file smoke test của Cấp độ 1), 2 epoch, chỉ đồ thị `train`: kiểm tra mã trong ~3 phút. **Không dùng số liệu này cho báo cáo.**
* `QUICK_RUN = False` — `assist2009` đầy đủ + `junyi` (toàn bộ người học, 200 bài tập phổ biến nhất), GRKT với đồ thị `train`, quỹ đạo mastery + tiêu chí hợp lý: **cấu hình cho báo cáo**. Ba công tắc ngoài phạm vi yêu cầu Cấp độ 2 (`RUN_GRAPH_ABLATIONS`, `RUN_CALIBRATION`, `RUN_MULTI_SEED`) đều mặc định `False`; bật lên nếu session còn dư thời lượng GPU.
* `MAX_SEQ_LEN = 200` ở chế độ báo cáo **để khớp đúng Cấp độ 1** (Cấp độ 1 train DKT/GKT với `MAX_SEQ_LEN=200`). Nhờ vậy checkpoint DKT nạp lại được đánh giá trên đúng cửa sổ nó đã học, và mục 4 đối chiếu AUC với 0.7866 là so sánh trực tiếp. Nếu hạ xuống 80 thì đường nền bị cắt ngắn hơn lúc train → AUC lệch là **lệch do cửa sổ, không phải do split**, và bước đối chiếu sẽ báo ⚠.
* GRKT **nặng hơn GKT Cấp độ 1 nhiều lần** (mỗi bước thời gian chạy vài chục phép nhân ma trận trên toàn đồ thị) và MỖI biến thể đồ thị là một lần train riêng, nên `RUN_GRAPH_ABLATIONS=True` tốn gấp ~4 thời gian. Đó là lý do mặc định chỉ train đồ thị `train`. Nếu vẫn vượt thời lượng Kaggle: giảm `EPOCHS` (5 → 3) trước, chỉ giảm `MAX_SEQ_LEN` khi thật sự bí.
* `RESUME_GRKT_CKPT = True` (mặc định) — nạp lại checkpoint của phiên trước thay vì train lại: `ckpt_GRKT-<graph>.pt` (dừng giữa epoch → train tiếp đúng batch còn lại, kèm optimizer + RNG) hoặc `GRKT-<graph>.pt` (đã train xong → chỉ đánh giá). Checkpoint có **fingerprint** (split/đồ thị/batch/`d_hidden`/`k_hidden`/seed/lr/…): resume với cấu hình khác sẽ **raise** thay vì lặng lẽ trộn hai lần train. Lần chạy đầu không có checkpoint nên cờ này vô hại.
* Ngân sách thời gian: `TIME_BUDGET_MIN` (600 phút) là ngân sách của cell §3, nhưng **bị chặn thêm bởi đồng hồ phiên Kaggle** (`SESSION_LIMIT_H = 12`, trừ `SESSION_SAFETY_MIN = 45` phút để đóng gói zip và tải về). Nhờ vậy dù bạn vào §3 muộn, kernel cũng dừng trước khi Kaggle cắt phiên. Hết ngân sách → checkpoint được ghi và cell §3 **tự tạo `level2_output.zip`**. Đặt `0` để tắt giới hạn.
* `MAX_USERS` chỉ ảnh hưởng tới tập phải chuẩn hoá (Junyi). `assist2009` dùng thẳng CSV đã commit nên luôn giữ đủ người học và giữ đúng split của Cấp độ 1.

In [ ]:
import os
import time
from pathlib import Path

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from level2.dataset_spec import ensure_canonical, get_spec, probe_columns, find_kaggle_input
from level2.experiment import run_level2

print("PyTorch:", torch.__version__, "| CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

QUICK_RUN = False   # True = chỉ chạy assist2009_smoke để kiểm tra mã; False = 2 tập đầy đủ

DATASETS = ["assist2009_smoke"] if QUICK_RUN else ["assist2009", "junyi"]
EPOCHS = 2 if QUICK_RUN else 5
BATCH_SIZE = 16 if QUICK_RUN else 32   # K=200 (toàn bộ Junyi sau cap khái niệm): B=32 để tensor
                                      # kích hoạt [B, K+1, k_hidden]×S bước vừa 15 GiB T4
# 200 = đúng cấu hình Cấp độ 1; đổi số khác thì mục 4 (đối chiếu DKT 0.7866) sẽ lệch.
MAX_SEQ_LEN = 50 if QUICK_RUN else 200
# None = TOÀN BỘ người học Junyi (≈72.630 sau lọc >=10 tương tác, ≈16,2 triệu lượt).
# Số người chỉ làm tăng thời gian train; VRAM phụ thuộc batch × (K+1), không phụ thuộc số người.
MAX_USERS = 300 if QUICK_RUN else None

# Cấp độ 2 chỉ yêu cầu: tái lập GRKT trên >=2 tập công khai với CÙNG split + CÙNG chỉ số,
# trực quan hoá mastery theo khái niệm/theo thời gian, và đo >=2 tiêu chí hợp lý của quỹ đạo.
# Ba núm dưới đây ngoài phạm vi đó nên mặc định TẮT; bật lại khi session còn dư thời lượng GPU.
RUN_BASELINES = True                    # DKT / GKT-Transition / GKT-Zero: nạp checkpoint Cấp độ 1, không train lại
# Ablation cấu trúc đồ thị là yêu cầu CẤP 1 (đã có kết quả trong kaggle_level1.ipynb).
# Bật True ở đây = train GRKT 4 lần cho mỗi tập (tốn gấp ~4 thời gian), chỉ để so sánh quỹ đạo.
RUN_GRAPH_ABLATIONS = False
VARIANTS = ("train", "dense", "zero", "shuffled") if RUN_GRAPH_ABLATIONS else ("train",)
RECORD_TRAJECTORY = True                # Nhiệm vụ 2 (BẮT BUỘC): m(u,c,t) + tiêu chí hợp lý
# YEU_CAU.md mục 7: Cấp độ 2 chỉ cần chọn MỘT phân tích nâng cao -> đã chọn quỹ đạo mastery.
RUN_CALIBRATION = False                 # ECE / Brier vẫn chạy được khi đặt True
# 3 seed của DKT/GKT đã chạy ở Cấp độ 1 trên đúng split này; GRKT nặng hơn nhiều lần.
RUN_MULTI_SEED = False
SEEDS = [42, 123, 456]
# Resume qua nhiều phiên Kaggle (xem §0.3 và §8.4). True: nạp GRKT-<graph>.pt (đã train xong ->
# chỉ đánh giá) hoặc ckpt_GRKT-<graph>.pt (dừng giữa epoch -> train tiếp đúng batch còn lại),
# thay vì train lại từ đầu. Lần chạy ĐẦU không có checkpoint nào nên cờ này vô hại; các lần sau
# nó là thứ giữ lại hàng giờ GPU. Checkpoint GRKT lưu fingerprint (split/đồ thị/batch/d_hidden/
# k_hidden/seed/lr...) và RNG: resume với cấu hình khác sẽ raise thay vì lặng lẽ trộn hai lần
# train, còn resume đúng cấu hình thì dòng ngẫu nhiên (gumbel_softmax) nối tiếp như chưa dừng.
# Muốn train lại từ đầu: đặt False VÀ xoá ckpt_GRKT-*.pt lẫn GRKT-*.pt.
RESUME_GRKT_CKPT = True

# Ngân sách thời gian cho CẢ cell §3 (phút). 600 = 10 giờ, chừa thời gian cho chuẩn hoá Junyi,
# quỹ đạo và §8.5 trước khi Kaggle cắt phiên. Hết ngân sách, GRKT ghi ckpt_GRKT-<graph>.pt rồi
# raise TrainingPaused và cell §3 TỰ ĐÓNG GÓI level2_output.zip — phần đã train không mất.
# 0/None = không giới hạn. Ngân sách còn bị chặn bởi đồng hồ phiên thật (xem khối dưới).
TIME_BUDGET_MIN = 30 if QUICK_RUN else 600
CKPT_EVERY_MIN = 10     # nhịp ghi checkpoint giữa epoch; session bị kill mất tối đa 10 phút GPU

# Phiên Kaggle bị cắt cứng sau SESSION_LIMIT_H giờ (một số tài khoản/docs ghi 9 giờ -> đặt 9).
# TIME_BUDGET_MIN tính từ lúc chạy cell §3 chứ không phải từ đầu phiên: nếu bạn đã dùng 3 giờ
# cho việc khác thì 600 phút sẽ vượt mốc và kernel bị kill giữa lúc train (mất phần chưa kịp
# đóng gói). Vì vậy ngân sách được chặn thêm bởi thời gian còn lại của phiên.
SESSION_LIMIT_H = 12
SESSION_SAFETY_MIN = 45   # chừa thời gian đóng gói zip + tải về trước khi phiên bị cắt

# Zip của phiên trước đã upload thành private Kaggle Dataset, vd
# "/kaggle/input/level2-resume/level2_output.zip" (hoặc thư mục đã giải nén — xem §0.3).
# None = tự dò /kaggle/input.
RESUME_ZIP = None

# Siêu tham số GRKT (mặc định theo bài báo; k_hidden nhỏ để bớt nặng trên T4)
D_HIDDEN, K_HIDDEN, K_HOP = 128, 16, 1
LR = 1e-3
PATIENCE = 2 if QUICK_RUN else 3
GRAPH_THRESH = 0.6      # ngưỡng nhị phân hoá rel/pre đếm từ train
# Ngưỡng lạnh: với tập smoke (15 người học) mọi cặp khái niệm đều hiếm, để 5 thì đồ thị train RỖNG
# và ablation vô nghĩa -> hạ xuống 1 để đồ thị train vẫn còn cạnh cho GRKT học.
COLD_THRESH = 1 if QUICK_RUN else 5
ALPHA = 0.01            # hệ số khoảng thời gian của GRKT
GAP_CAP_MIN = 240       # chặn Δt (phút) — xem level2/grkt_data._cumulative_times

USE_COMMITTED_CKPTS = True   # nạp DKT/GKT đã train ở Cấp độ 1 thay vì train lại
BASELINE_EPOCHS = 2 if QUICK_RUN else 20   # epoch khi PHẢI train lại đường nền (tập chưa có ckpt Cấp độ 1): 20 = đúng công thức Cấp độ 1
SPLIT_SEED = 42              # BẤT DI BẤT DỊCH: đúng seed split của Cấp độ 1

SAVE_DIR = Path("/kaggle/working/level2_out") if IS_KAGGLE else (REPO / "outputs" / "level2")
LEVEL1_CKPT_DIR = REPO / "outputs" / "level1"
SAVE_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH = Path("/kaggle/working/level2_output.zip") if IS_KAGGLE else (REPO / "level2_output.zip")


def _session_start_epoch():
    """Mốc bắt đầu phiên Kaggle = lúc PID 1 (container) khởi động; None nếu không đọc được."""
    try:
        fields = Path("/proc/1/stat").read_text().rsplit(")", 1)[1].split()
        start_ticks = float(fields[19])                       # field 22 của /proc/<pid>/stat
        uptime = float(Path("/proc/uptime").read_text().split()[0])
        return time.time() - uptime + start_ticks / os.sysconf("SC_CLK_TCK")
    except Exception:
        return None

_session_start = _session_start_epoch() if IS_KAGGLE else None
if TIME_BUDGET_MIN and _session_start is not None:
    _elapsed_min = (time.time() - _session_start) / 60.0
    _left_min = SESSION_LIMIT_H * 60 - _elapsed_min - SESSION_SAFETY_MIN
    if not (0.0 <= _elapsed_min <= SESSION_LIMIT_H * 60):
        # PID 1 không phải tiến trình mở đầu phiên (vd. sidecar sống lâu hơn giới hạn): bỏ
        # clamp thay vì hạ ngân sách còn 1 phút và dừng ngay sau batch đầu.
        print(f"[cảnh báo] đồng hồ phiên không hợp lệ (đã chạy {_elapsed_min:.0f} phút) -> "
              "bỏ qua chặn theo đồng hồ, TIME_BUDGET_MIN tính từ lúc chạy cell §3.")
    else:
        print(f"Phiên Kaggle: đã chạy {_elapsed_min:.0f} phút | còn {_left_min:.0f} phút tới chốt "
              f"{SESSION_LIMIT_H}h (đã trừ {SESSION_SAFETY_MIN} phút an toàn cho zip)")
        if _left_min < 60:
            print("[CẢNH BÁO] còn dưới 1 giờ: lần chạy này chỉ kịp đánh giá checkpoint có sẵn — "
                  "hãy attach zip của phiên trước (RESUME_ZIP, §0.3) và giữ RESUME_GRKT_CKPT=True.")
        TIME_BUDGET_MIN = max(1.0, min(float(TIME_BUDGET_MIN), _left_min))
elif TIME_BUDGET_MIN:
    print("Không đọc được đồng hồ phiên -> TIME_BUDGET_MIN tính từ lúc chạy cell §3.")

print(f"Config: datasets={DATASETS} epochs={EPOCHS} batch={BATCH_SIZE} max_seq_len={MAX_SEQ_LEN}")
print(f"        variants={VARIANTS} multiseed={RUN_MULTI_SEED} trajectory={RECORD_TRAJECTORY} calibration={RUN_CALIBRATION}")
print(f"Ngân sách cell §3: {TIME_BUDGET_MIN:.0f} phút | zip resume: {ZIP_PATH}")
print("Output dir :", SAVE_DIR)
print("Level-1 ckpts:", LEVEL1_CKPT_DIR, LEVEL1_CKPT_DIR.is_dir())

### 0.3 Khôi phục phiên train bị ngắt

Kaggle cắt phiên sau 12 giờ và xoá `/kaggle/working` khi phiên kết thúc, nên checkpoint chỉ sống qua phiên nếu được **tải về**. Quy trình hai phiên:

1. **Phiên trước**: khi hết ngân sách, cell §3 tự ghi checkpoint **và tự tạo `level2_output.zip`** (chứa `ckpt_GRKT-*.pt`, `GRKT-*.pt`, `baselines/ckpt_*.pt`, `split.json`) rồi dừng với `TrainingPaused`. **Tải zip đó về** bằng tab *Output* / sidebar *Files* **trước khi phiên bị cắt** — Kaggle không tự lưu output của phiên interactive. (Nếu phiên trước kết thúc bình thường, chạy **§8.5** để đóng gói.)
2. **Phiên này**: Kaggle → *Datasets* → *New Dataset* → upload file zip (đặt **private**) → trong notebook *Add Input* dataset đó.
3. Đặt `RESUME_ZIP` ở §0.2 (hoặc để `None` — cell dưới tự dò), rồi chạy cell này **trước §1**.
4. Chạy lại từ §1: GRKT đã train xong được nạp và chỉ đánh giá; GRKT dừng giữa epoch train tiếp từ đúng batch/optimizer/RNG; **đường nền DKT/GKT phải train lại** (Junyi) cũng nối tiếp theo epoch từ `baselines/ckpt_<tên>.pt`. Các checkpoint có fingerprint nên resume sai split/cấu hình sẽ dừng ngay thay vì cho số liệu không tái lập.

> **Kaggle tự giải nén zip trong dataset** khi bạn attach. Vì vậy `level2_output.zip` sau khi upload thường xuất hiện dưới dạng `/kaggle/input/<tên-dataset>/level2_out/<dataset>/…` chứ không còn là file `.zip`. Cell dưới xử lý **cả hai** dạng; `RESUME_ZIP` có thể trỏ vào file `.zip` hoặc thẳng thư mục đã giải nén.

Cell này chỉ sao chép đúng cây `level2_out/<dataset>/…` vào `/kaggle/working` (bỏ qua file nào ở đích **mới hơn hoặc bằng** nguồn, nên chạy lại giữa phiên không ghi đè checkpoint vừa train), và không chép `junyi_canonical.csv` (zip mặc định bỏ file đó — `ensure_canonical()` tạo lại, 15–30 phút).

In [ ]:
# ==== 0.3 Khôi phục checkpoint phiên trước (zip HOẶC cây đã giải nén của Kaggle) ====
import time
import zipfile

TOP = SAVE_DIR.name          # "level2_out" trên Kaggle, "level2" khi chạy local


def _is_newer_or_equal(target: Path, src_mtime: float) -> bool:
    # Chạy lại cell này trong CÙNG phiên không được ghi đè checkpoint mới hơn bằng bản cũ
    # trong zip/dataset.
    return target.exists() and target.stat().st_mtime >= src_mtime


def _restore_from_zip(path: Path) -> int:
    n = 0
    with zipfile.ZipFile(path) as zf:
        infos = [i for i in zf.infolist() if not i.is_dir()]
        # extractall không chặn zip-slip; kiểm tra trước vì file được tải lên từ máy khác.
        unsafe = [i.filename for i in infos if i.filename.startswith("/") or ".." in Path(i.filename).parts]
        if unsafe:
            raise RuntimeError(f"zip chứa đường dẫn không an toàn: {unsafe[:5]}")
        top = TOP + "/"
        if not any(i.filename.startswith(top) for i in infos):
            raise RuntimeError(f"{path} không phải zip do §8.5 tạo (thiếu tiền tố {top})")
        for info in infos:
            target = SAVE_DIR / Path(info.filename).relative_to(TOP)
            if _is_newer_or_equal(target, time.mktime(info.date_time + (0, 0, -1))):
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(info) as src, open(target, "wb") as dst:
                shutil.copyfileobj(src, dst)
            n += 1
    return n


def _restore_from_tree(src: Path) -> int:
    n = 0
    for p in src.rglob("*"):
        if not p.is_file():
            continue
        target = SAVE_DIR / p.relative_to(src)
        if _is_newer_or_equal(target, p.stat().st_mtime):
            continue
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(p, target)   # copy2 giữ mtime -> lần chạy sau vẫn so sánh được
        n += 1
    return n


# Kaggle TỰ GIẢI NÉN mọi zip trong dataset được attach, nên "upload level2_output.zip" thường
# xuất hiện dưới dạng /kaggle/input/<dataset>/level2_out/<dataset>/... chứ không còn file .zip.
# Cả hai dạng đều được xử lý; RESUME_ZIP có thể trỏ vào file .zip hoặc thẳng thư mục.
_resume_zip, _resume_tree = None, None
if RESUME_ZIP:
    _p = Path(RESUME_ZIP)
    if not _p.exists():
        raise FileNotFoundError(f"RESUME_ZIP={RESUME_ZIP} không tồn tại — kiểm tra Add Input")
    _resume_zip, _resume_tree = (_p, None) if _p.is_file() else (None, _p)
else:
    _root = Path("/kaggle/input")
    _zips = sorted(_root.glob("*/level2_output.zip")) if _root.is_dir() else []
    _trees = sorted(p for p in _root.rglob(TOP) if p.is_dir()) if _root.is_dir() else []
    if len(_zips) == 1:
        _resume_zip = _zips[0]
    elif len(_zips) > 1:
        print("Có nhiều zip trong /kaggle/input — đặt RESUME_ZIP cho rõ ràng:")
        for c in _zips:
            print("   ", c)
    elif len(_trees) == 1:
        _resume_tree = _trees[0]
    elif len(_trees) > 1:
        print(f"Có nhiều thư mục {TOP} trong /kaggle/input — đặt RESUME_ZIP cho rõ ràng:")
        for c in _trees:
            print("   ", c)

if _resume_zip is not None:
    print(f"Đã phục hồi {_restore_from_zip(_resume_zip)} file từ zip {_resume_zip}")
elif _resume_tree is not None:
    print(f"Đã phục hồi {_restore_from_tree(_resume_tree)} file từ cây {_resume_tree} "
          "(Kaggle đã tự giải nén zip khi upload)")
else:
    print("Không có zip/cây checkpoint nào trong /kaggle/input -> lần chạy này train từ đầu.")

# Liệt kê checkpoint tìm thấy để biết phiên trước dừng ở đâu (§8.4 in lại bất kỳ lúc nào).
found = sorted(SAVE_DIR.rglob("ckpt_*.pt")) + sorted(SAVE_DIR.rglob("GRKT-*.pt"))
if not found:
    print("Không có checkpoint GRKT/đường nền -> lần chạy này train từ đầu.")
for c in found:
    try:
        payload = torch.load(c, map_location="cpu", weights_only=False)
    except Exception as exc:
        print(f"  {c.relative_to(SAVE_DIR.parent)} | không đọc được: {exc}")
        continue
    fmt = payload.get("format") if isinstance(payload, dict) else None
    if fmt in ("grkt-resume-v1", "level1-resume-v1"):
        print(f"  {c.relative_to(SAVE_DIR.parent)} | epoch {payload.get('epoch')}, batch "
              f"{payload.get('batch_pos', 0)} | best val AUC={float(payload.get('best_auc', -1)):.4f} "
              f"(train tiếp được)")
    elif fmt == "grkt-final-v1":
        print(f"  {c.relative_to(SAVE_DIR.parent)} | GRKT đã train xong (có fingerprint) "
              f"-> chỉ đánh giá, không train lại")
    else:
        print(f"  {c.relative_to(SAVE_DIR.parent)} | checkpoint cuối (không fingerprint: không "
              f"kiểm chứng được split, sẽ cảnh báo khi nạp)")

## 1. GRKT làm gì, và notebook này đã sửa gì so với bản gốc

**Cơ chế (mỗi bước tương tác i):**

1. *Knowledge retrieval*: trạng thái mastery của mọi khái niệm lan truyền trên `rel_map` (đồng xuất hiện) và `pre_map` (tiên quyết, kèm `sub_map = pre_mapᵀ`) → `m̃`; dự đoán `P(đúng) = σ(m̃(c_i) − δ(q_i))`.
2. *Memory strengthening*: làm đúng thì **cộng** `gain`, làm sai thì **trừ** `loss` vào trạng thái của khái niệm vừa làm (và lan toả theo cạnh đồ thị).
3. *Learning–forgetting*: giữa hai bước, `exp(−(n+1)·Δt·k)` quyết định khái niệm nào được học thêm và khái niệm nào phai đi theo **thời gian thực**.

**Toàn bộ chỉnh sửa so với mã upstream** (được liệt kê lại trong `KE_HOACH.md`):

| # | Chỗ sửa | Lý do |
|---|---|---|
| 1 | `GRKT/` thành Python package, `import utils/metric/BaseModel` → import tương đối | `GKT/` (Cấp độ 1) cũng có `utils.py`; nếu không sửa thì nạp nhầm module của nhau |
| 2 | Bỏ `GRKT/Runner.py`: tự vòng lặp epoch trong `level2/grkt_api.py` | Runner cắt chuỗi thành cửa sổ 100 và chia **k-fold trên cửa sổ** ⇒ cùng một người học nằm ở cả train lẫn test (phạm §6.4). Mỗi người học giờ là MỘT chuỗi, split lấy từ Cấp độ 1 |
| 3 | Dựng lại `rel_map`/`pre_map` từ **train only** (`level2/grkt_data.build_relevance_maps`) | File `assist09.bin` gốc đếm đồ thị trên toàn bộ dữ liệu ⇒ rò rỉ (phạm §6.2) |
| 4 | Trục thời gian = **phút cộng dồn, chặn trên 240 phút** | `exp(−(n+1)·Δt·k)` với Δt thô (order_id/µs) underflow về 0 ⇒ mô hình "quên sạch" sau 1 bước, AUC chỉ bằng tỉ lệ đúng |
| 5 | `args.thresh = 0.5` và ma trận đưa vào đã nhị phân | Nhánh `if thresh == 0` của upstream gán **toàn bộ** ma trận = 1, kể cả hàng/cột padding số 0 |
| 6 | Thêm hook `rec` trong `GRKT/model/GRKT.py::forward` (3 điểm ghi) | Cấp master `m_pre/m_tilde/m_strong` cho quỹ đạo mastery; `rec is None` khi huấn luyện ⇒ không đổi hành vi, không thêm chi phí |
| 7 | `GRKT/Training.py`: `eval('GRKT.GRKT')` → registry tường minh, chặn dataset theo allowlist | bỏ thực thi chuỗi thành mã (bảo mật) và chặn nạp `assist09.bin` rò rỉ |

**Không sửa**: kiến trúc mô hình, công thức loss (BCE), `get_feed_dict`/`feature2tensor` (đệm batch), định nghĩa AUC/ACC. Vì vậy số liệu GRKT ở đây so sánh trực tiếp được với DKT/GKT của Cấp độ 1.

## 2. EDA và kiểm tra cột dữ liệu (bắt buộc trước khi chuẩn hoá Junyi)

Cell dưới in ra **tên cột thật** của từng tập dữ liệu. `level2/dataset_spec.py` khớp tên cột theo một danh sách ứng viên; nếu Kaggle đổi tên cột, nó sẽ **dừng và in ra các cột có sẵn** thay vì đoán mò — khi đó chỉ cần bổ sung tên cột vào `JUNYI.columns` rồi chạy lại.

**Junyi — ánh xạ cột của bản Kaggle đang dùng** (`Log_Problem.csv`, 3,0 GB, 16,2 triệu dòng):

| Cột chuẩn hoá | Cột gốc | Ghi chú |
|---|---|---|
| `user_id` | `uuid` | 72.630 người học (khoá của `Info_UserData.csv`) |
| `skill_id` | `ucid` | bài tập/khái niệm (1.326 giá trị) — **đã có trong log**, không cần nối bảng tra; file **không có** cột tên `exercise` |
| `problem_id` | `upid` | 25.785 câu hỏi |
| `correct` | `is_correct` | đúng ngay lần đầu **và** không dùng gợi ý |
| `timestamp_ms` | `timestamp_TW` | UTC+8, làm tròn 15 phút (Δt trong một phiên thường = 0) |
| `ms_first_response` | `total_sec_taken` | đơn vị **giây**, đổi sang ms |

Log chuẩn hoá in đúng ánh xạ đã chọn, nên nếu lần chạy sau Kaggle đổi schema thì nhìn dòng `ánh xạ cột:` là biết ngay cột nào bị thay.

**Chạy toàn bộ người học:** cấu hình báo cáo đặt `MAX_USERS = None`, tức giữ mọi người học có
≥ 10 tương tác (≈72.630 người / 16,2 triệu lượt); vẫn giữ 200 bài tập phổ biến nhất vì đồ thị
GRKT là dense theo khái niệm. `ensure_canonical` ghi kèm `junyi_canonical.csv.meta.json` chứa
đúng `max_users`/`min_interactions`/`top_concepts` đã dùng: nếu session hiện tại đã chuẩn hoá với
cap cũ (ví dụ QUICK_RUN 300 người) rồi bạn đổi sang toàn bộ, nó sẽ **dừng và yêu cầu xoá** file cũ
thay vì lặng lẽ dùng lại; file cũ không có meta thì chỉ cảnh báo, khi đó xoá
`level2_out/junyi/junyi_canonical.csv` trước khi chạy lại.

Junyi là dataset **có điều khoản** (`requiresSubscription`): phải bấm **Agree/Subscribe** trên trang dataset *trước*, rồi mới `Add input`. Nếu quên bước này, `/kaggle/input` không có gì và cell sẽ in hiện trạng thư mục input kèm đường dẫn dataset cần attach, thay vì chỉ báo FileNotFoundError. Files bên trong: `Log_Problem.csv` (16.217.311 dòng / 72.630 người học), `Info_Content.csv`, `Info_UserData.csv`.


In [ ]:
from level2.dataset_spec import list_kaggle_inputs

for name in DATASETS:
    spec = get_spec(name)
    print("=" * 72)
    print(f"[{name}] {spec.note}")
    if name == "junyi":
        guess = find_kaggle_input(patterns=("junyi", "log_problem"))
        print("  file gốc dò được trong /kaggle/input:", guess)
        if guess is None:
            print("  Hiện trạng /kaggle/input:")
            for line in list_kaggle_inputs().splitlines():
                print("    " + line)
            print("  >> Sửa: mở https://www.kaggle.com/datasets/junyiacademy/learning-activity-public-dataset-by-junyi-academy")
            print("     bấm Agree/Subscribe (dataset này bắt buộc subscribe — không subscribe thì Add input không có tác dụng),")
            print("     rồi ở notebook: Add input > Datasets > tìm 'junyiacademy'. Attach xong chạy lại cell này.")
            print("  Cell này BỎ QUA junyi để còn xem EDA của tập khác; cell thực nghiệm (§3) vẫn sẽ dừng nếu thiếu dữ liệu.")
            continue
        else:
            # In header THẬT của file gốc: tên cột đổi thì thấy ngay ở đây, bổ sung vào
            # JUNYI.columns trong level2/dataset_spec.py thay vì để cell chuẩn hoá đoán mò.
            with open(guess, encoding="utf-8-sig", errors="replace") as fh:
                header = fh.readline().strip()
            print(f"  cột gốc của {guess.name}: {header}")
    try:
        canonical = ensure_canonical(spec, out_dir=SAVE_DIR / name, max_users=MAX_USERS, verbose=True)
    except FileNotFoundError as exc:
        print(f"  ⚠ Bỏ qua {name} trong EDA vì thiếu dữ liệu:\n{exc}")
        continue
    print("  CSV chuẩn hoá:", canonical)
    print("  cột chuẩn hoá:", probe_columns(canonical))

    eda = pd.read_csv(canonical)
    print(f"  {len(eda):,} tương tác | {eda['user_id'].nunique():,} người học | "
          f"{eda['skill_id'].nunique():,} khái niệm | {eda['problem_id'].nunique():,} câu hỏi | "
          f"tỉ lệ đúng {eda['correct'].mean():.3f}")

    fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
    # Vẽ bằng matplotlib thuần: pandas 3 đã bỏ accessor .hist()/.barh() trực tiếp trên Series
    axes[0].hist(eda.groupby("user_id").size().to_numpy(), bins=40, edgecolor="black", alpha=0.7)
    axes[0].set_title(f"{name}: số tương tác mỗi người học")
    axes[0].set_xlabel("số lượt")
    top_skill = eda["skill_id"].value_counts().head(20).iloc[::-1]
    axes[1].barh([str(x) for x in top_skill.index], top_skill.to_numpy(), color="#4878a6")
    axes[1].set_title(f"{name}: 20 khái niệm thường gặp nhất")
    axes[1].set_xlabel("số lượt")
    fig.tight_layout()
    plt.show()


## 3. Chuỗi thực nghiệm Cấp độ 2

`run_level2(...)` gọi `run_dataset(...)` cho từng tập dữ liệu. Với cấu hình mặc định (chỉ chạy đúng những gì Cấp độ 2 yêu cầu), **B3/B4 chỉ sinh và train đồ thị `train`**, còn **B7 (calibration)** và **B8 (đa seed)** bị bỏ qua — bật `RUN_GRAPH_ABLATIONS` / `RUN_CALIBRATION` / `RUN_MULTI_SEED` nếu muốn chạy chúng:

| Bước | Nội dung | Sản phẩm |
|---|---|---|
| **B1** | chuẩn hoá CSV + split theo người học (seed 42, dùng lại pipeline Cấp độ 1) | `split.json` |
| **B2** | dựng chuỗi GRKT từ cùng file và **đối chiếu từng người học** với bundle Cấp độ 1 (khớp thì mới chạy tiếp) | log |
| **B3** | đếm `rel_map`/`pre_map` **chỉ từ train** (+ sinh dense/zero/shuffled nếu bật ablation) | `graphs_train_only.npz`, `leak_report.txt` |
| **B4** | huấn luyện GRKT cho từng biến thể được bật, early stop theo **val AUC toàn cục**; checkpoint giữa epoch mỗi `CKPT_EVERY_MIN` phút | `GRKT-<graph>.pt`, `ckpt_GRKT-<graph>.pt` |
| **B5** | đường nền DKT / GKT-Transition / GKT-Zero trên cùng split (nạp checkpoint Cấp độ 1 nếu có; phải train lại thì checkpoint cuối mỗi epoch) | `level2_metrics.csv`, `baselines/*.pt` |
| **B6** | ghi quỹ đạo m(u,c,t) + 3 tiêu chí hợp lý | `traj_<graph>.npz`, `mastery_<graph>.csv`, `heatmap/curves`, `trajectory_metrics.csv` |
| **B7** | *(tuỳ chọn, mặc định tắt)* ECE / Brier / reliability cho mọi mô hình | `calibration.csv`, `reliability.png` |
| **B8** | *(tuỳ chọn, mặc định tắt — đa seed đã có ở Cấp độ 1)* split cố định, chỉ đổi khởi tạo + thứ tự batch | `multiseed_metrics.csv` |

`TIME_BUDGET_MIN` (mặc định 600 phút) là ngân sách của **cả cell này**, tính một lần cho mọi tập — nhưng bị **chặn trên bởi đồng hồ phiên Kaggle**: cell §0.2 đọc thời điểm session bắt đầu (`/proc/1/stat`) rồi hạ ngân sách xuống còn `12 giờ − 45 phút` dự phòng, nên `600` phút không bao giờ vượt quá phần thời gian còn lại.

Hết ngân sách, bước đang chạy ghi checkpoint rồi raise `TrainingPaused`: cell hiện banner, **tự đóng gói `level2_output.zip`** ngay (không cần đợi §8.5) rồi dừng — phần đã train không mất; phiên sau chạy tiếp theo quy trình §0.3. Đặt `0`/`None` = không giới hạn theo ngân sách (nhưng Kaggle vẫn có thể kill session, nên zip tự động mới là thứ bảo vệ kết quả — nhớ tải nó về).


In [ ]:
from level2.archive import package_output
from level2.grkt_api import TrainingPaused


def _package_now(reason: str) -> None:
    """Đóng gói checkpoint hiện có NGAY — Kaggle xoá /kaggle/working khi phiên kết thúc."""
    try:
        package_output(SAVE_DIR, ZIP_PATH)
        print(f"\n>> Đã tạo {ZIP_PATH} ({reason}). TẢI FILE NÀY VỀ trước khi phiên kết thúc:")
        print("   tab Output (hoặc sidebar Files) -> level2_output.zip -> tải xuống.")
    except Exception as zexc:  # noqa: BLE001 - không được che lỗi gốc của lần train
        print(f"\n[cảnh báo] không đóng gói được {ZIP_PATH}: {type(zexc).__name__}: {zexc}\n"
              f"  -> chạy cell §8.5 thủ công; checkpoint vẫn nằm trong {SAVE_DIR}")


try:
    result = run_level2(
        DATASETS,
        save_root=SAVE_DIR,
        time_budget_min=TIME_BUDGET_MIN,
        ckpt_every_min=CKPT_EVERY_MIN,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        max_seq_len=MAX_SEQ_LEN,
        lr=LR,
        patience=PATIENCE,
        d_hidden=D_HIDDEN,
        k_hidden=K_HIDDEN,
        k_hop=K_HOP,
        alpha=ALPHA,
        graph_thresh=GRAPH_THRESH,
        cold_thresh=COLD_THRESH,
        gap_cap_min=GAP_CAP_MIN,
        max_users=MAX_USERS,
        variants=VARIANTS,
        split_seed=SPLIT_SEED,
        run_baselines=RUN_BASELINES,
        baseline_ckpt_dir=(LEVEL1_CKPT_DIR if USE_COMMITTED_CKPTS else None),
        baseline_epochs=BASELINE_EPOCHS,
        allow_retrain_baselines=True,
        record_trajectory=RECORD_TRAJECTORY,
        run_calibration=RUN_CALIBRATION,
        run_multi_seed=RUN_MULTI_SEED,
        seeds=SEEDS,
        resume_grkt_ckpt=RESUME_GRKT_CKPT,
        device=device,
    )
except TrainingPaused as exc:
    print("\n" + "!" * 78)
    print("DỪNG CHỦ ĐỘNG THEO TIME_BUDGET_MIN — checkpoint đã ghi, KHÔNG mất phần đã train.")
    print(" ", exc)
    print("! " * 39)
    _package_now("dừng theo ngân sách thời gian")
    raise
except Exception:
    # Lỗi bất ngờ giữa chừng (OOM, mất mạng khi lưu, ...): vẫn giữ lại những gì đã train.
    _package_now("lỗi giữa chừng — gói checkpoint để phiên sau resume")
    raise

combined = result["combined"]
print("\nSố dòng trong bảng nối:", len(combined))
print("Chạy §8.5 để đóng gói toàn bộ sản phẩm (zip cũng là gói resume) trước khi tắt phiên.")
combined

## 4. Bảng so sánh DKT / GKT / GRKT và kiểm tra đối chiếu split

In [ ]:
display(combined[["dataset", "model", "graph", "auc", "acc", "loss", "val_auc", "n_test"]])

# ---------------------------------------------------------------- kiểm tra đối chiếu split
# Đường nền DKT/GKT ở đây KHÔNG được train lại: checkpoint lấy từ outputs/level1/. Nếu Cấp độ 2
# đang thật sự dùng lại đúng split của Cấp độ 1 (cùng file, cùng seed 42, cùng max_seq_len), AUC
# đánh giá lại phải khớp số đã công bố trong vòng vài chữ số thập phân. Đây là bằng chứng thực
# nghiệm của yêu cầu "một split duy nhất cho mọi mô hình" (YEU_CAU.md §6.3).
LEVEL1_MAX_SEQ_LEN = 200
ref_file = LEVEL1_CKPT_DIR / "level1_metrics.csv"
ref = None
if ref_file.exists():
    ref = pd.read_csv(ref_file).rename(columns={"Mô hình": "model", "Test AUC": "ref_auc"})
    if MAX_SEQ_LEN != LEVEL1_MAX_SEQ_LEN:
        print(f"[cảnh báo] MAX_SEQ_LEN={MAX_SEQ_LEN} khác Cấp độ 1 ({LEVEL1_MAX_SEQ_LEN}): "
              "chuỗi eval bị cắt ngắn hơn lúc train checkpoint, lệch AUC là do cửa sổ cắt chuỗi. "
              "Đặt MAX_SEQ_LEN=200 nếu muốn đối chiếu khớp chính xác với Cấp độ 1.")
    tol = 0.02 if MAX_SEQ_LEN != LEVEL1_MAX_SEQ_LEN else 0.005
    for name in ["DKT", "GKT-Transition", "GKT-Zero"]:
        row = combined[(combined["dataset"] == "assist2009") & (combined["model"] == name)]
        rr = ref[ref["model"] == name]
        if len(row) and len(rr):
            a2, a1 = float(row["auc"].iloc[0]), float(rr["ref_auc"].iloc[0])
            flag = "OK (cùng split)" if abs(a2 - a1) <= tol else f"lệch > {tol} — kiểm tra lại split và max_seq_len"
            print(f"{name:16s} | Cấp độ 1 = {a1:.4f} | Cấp độ 2 = {a2:.4f} | Δ={a2 - a1:+.4f}  {flag}")
else:
    print("Không có outputs/level1/level1_metrics.csv — bỏ qua bước đối chiếu.")

if "assist2009" not in set(combined["dataset"]):
    print("[đối chiếu] Lần chạy này không có assist2009 (chế độ smoke dùng assist2009_smoke, "
          "hoặc chỉ chạy junyi) nên KHÔNG kiểm tra được AUC đường nền với Cấp độ 1. "
          "Bắt buộc chạy lại với DATASETS có 'assist2009' trước khi đưa số liệu vào báo cáo.")

fig, ax = plt.subplots(figsize=(9, 4))
sub = combined[combined["dataset"] == combined["dataset"].iloc[0]]
# .tolist() để so sánh bằng str thông thường, không phụ thuộc dtype của cột
graphs = sub["graph"].tolist()
bars = ax.bar(sub["model"], sub["auc"], color=["#9aa5b1" if g == "N/A" else "#4878a6" for g in graphs])
ax.set_ylabel("Test AUC")
ax.set_title(f"Test AUC trên {sub['dataset'].iloc[0]} — cùng một split, cùng một chỉ số")
ax.set_ylim(min(0.5, float(sub["auc"].min()) - 0.05), 0.85)
axhline_val = float(ref["ref_auc"][ref["model"] == "DKT"].iloc[0]) if ref is not None else 0.7866
ax.axhline(axhline_val, ls="--", lw=1, color="#c0392b", label=f"DKT Cấp độ 1 = {axhline_val:.4f}")
for b, v in zip(bars, sub["auc"]):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.004, f"{v:.4f}", ha="center", fontsize=8)
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

## 5. Quỹ đạo mastery m(u, c, t) và các tiêu chí hợp lý

**Định nghĩa ba mốc đo trong một bước** (xem `level2/trajectory.py`):

* `m_pre[i]` — mastery của mọi khái niệm **trước** cập nhật ở bước i: đúng là trạng thái mô hình đã dùng để dự đoán.
* `m̃[i]` — mastery **sau giai đoạn truy xuất** (đã lan truyền trên cạnh đồ thị): khác `m_pre[i]` đúng ở phần lan truyền.
* `m_strong[i]` — mastery **sau giai đoạn củng cố** (đúng → cộng gain, sai → trừ loss).

Suy ra mỗi bước tách được hai cơ chế: `Δm_củng_cố = m_strong[i] − m_pre[i]` (ảnh hưởng của việc làm đúng/sai) và `Δm_học_quên = m_pre[i+1] − m_strong[i]` (ảnh hưởng của học + quên theo thời gian).

**Bốn tiêu chí hợp lý** (đề bài yêu cầu ≥ 2; cả bốn đều đo được trên đồ thị `train` nên cấu hình mặc định đã đủ):

1. `consistency_strengthen` — tại khái niệm vừa làm: đúng thì mastery phải tăng, sai thì phải giảm. Bất đẳng thức này đúng *theo cấu trúc* của GRKT, nên nó là **kiểm chứng cài đặt**: nếu ≠ 1 thì hook ghi quỹ đạo hoặc tiền xử lý đã sai.
2. `consistency_spread` — tại các khái niệm **láng giềng 1-hop** của khái niệm vừa làm (không kể nó): một lần làm bài có kéo theo khái niệm liên quan hay không. Đây là tiêu chí **có thể sai thật**, không chỉ là kiểm chứng cài đặt.
3. `locality` — tỉ lệ lượng thay đổi mastery nằm trong {đang làm ∪ láng giềng}. Cao = mô hình không "làm bẩn" những khái niệm không liên quan.
4. `forget_slope` — hồi quy Δm giữa hai bước theo Δt tại các khái niệm không bị chạm; kỳ vọng ≤ 0. Độ dốc dương nghĩa là mô hình cho rằng "không làm gì" cũng giỏi thêm.

Toàn bộ các biến thể đồ thị được đo trên **cùng một tập láng giềng** (của đồ thị `train`), nên các cột so sánh được với nhau.
Các câu so sánh chéo (`train` phải khác `shuffled`/`dense`/`zero`, `zero` có locality = 1.0 tầm thường, `|Δm| retrieval` của `zero` phải ~0) **chỉ xuất hiện khi bật `RUN_GRAPH_ABLATIONS=True`**, vì khi đó mỗi biến thể là một lần train riêng. Với cấu hình mặc định, bảng chỉ có dòng `train` — vẫn thoả yêu cầu "đo ≥ 2 tiêu chí hợp lý" của đề bài và không có cột nào để so sánh chéo.

> **Riêng Junyi — đọc `forget_slope` và `mean_abs_delta_learn_forget` cho đúng:** `timestamp_TW` được làm tròn 15 phút, nên các lượt liên tiếp trong cùng một phiên có Δt = 0. Khi đó `exp(−(n+1)·0·k) = 1` (không quên) và `1 − 1 = 0` (không học thêm): hai đại lượng này **bằng 0 vì dữ liệu không có mốc thời gian mịn hơn**, không phải vì cơ chế hỏng. Muốn đo suy giảm thật trên Junyi phải lấy mốc thời gian mịn hơn (`total_sec_taken`, đang ghi ở cột `ms_first_response`), không dùng timestamp 15 phút — ghi rõ giới hạn này khi báo cáo thay vì coi số 0 là kết quả.


In [ ]:
for name, run in result["runs"].items():
    tt = run.get("trajectory_table")
    if tt is None or len(run.get("trajectories", {})) == 0:
        print(f"[{name}] RECORD_TRAJECTORY = False, bỏ qua.")
        continue
    kind0 = sorted(run["trajectories"])[0]
    n_users = len(run["trajectories"][kind0]["users"])
    print("=" * 72)
    print(f"[{name}] TIÊU CHÍ HỢP LÝ CỦA QUỸ ĐẠO (đo trên {n_users} người học thuộc tập test)")
    print("=" * 72)
    display(tt)
    print("Giải thích: spread = None nghĩa là biến thể đó KHÔNG lan truyền sang láng giềng "
          "(đồ thị zero), không phải 'tệ' — hãy đọc cùng cột locality.\n")

### 5.1 Thay đổi mastery đến từ đâu (phân tách ba giai đoạn của GRKT)

GRKT cập nhật trạng thái theo ba bước: **retrieval** (đọc trạng thái các đỉnh kề sang khái
niệm đang làm), **strengthening** (tăng/giảm độ thành thạo theo kết quả vừa làm) rồi
**learn/forget** (lan truyền phần tăng đó sang láng giềng và suy giảm theo thời gian). Bảng
dưới đo `|Δm|` trung bình của từng giai đoạn trên cùng các người học, cho mỗi biến thể đồ thị
được bật. Với cấu hình mặc định chỉ có dòng `train`; ba gạch đầu dòng so sánh dưới đây cần
`RUN_GRAPH_ABLATIONS=True` (mỗi biến thể là một lần train riêng):

* `mean_abs_delta_retrieval` của biến thể `zero` **phải xấp xỉ 0** — không có cạnh thì không
  có gì để đọc. Nếu nó > 0 thì đồ thị `zero` đang không thật sự rỗng (lỗi tiền xử lý).
* `mean_abs_delta_strengthen` gần như không đổi giữa các biến thể — giai đoạn này chỉ nhìn kết
  quả làm bài của chính khái niệm đang trả lời.
* `mean_abs_delta_learn_forget` khác 0 ở MỌI biến thể (suy giảm theo thời gian là cơ chế nội
  tại). Tác động của đồ thị nằm ở CHÊNH LỆNH giữa `train`/`dense` và `zero`, không phải ở giá
  trị tuyệt đối.

In [ ]:
from level2.trajectory import stage_attribution

rows = []
for name, run in result["runs"].items():
    for kind, traj in sorted(run.get("trajectories", {}).items()):
        st = stage_attribution(traj)
        st.update({"dataset": name, "graph": kind})
        rows.append(st)
if rows:
    stage_df = pd.DataFrame(rows)
    cols = ["dataset", "graph"] + [c for c in stage_df.columns if c not in ("dataset", "graph")]
    display(stage_df[cols])
    zero = stage_df[stage_df["graph"] == "zero"]
    if len(zero):
        z = float(zero["mean_abs_delta_retrieval"].iloc[0])
        print(f"kiểm tra đồ thị rỗng: |Δm| retrieval của biến thể zero = {z:.4f} "
              + "(phải ~0; nếu > 0 thì graph_variant('zero') chưa thật sự không cạnh)")
    print("Với các biến thể có cạnh, so mean_abs_delta_learn_forget giữa 'train' và 'zero': "
          "chênh lệch đó chính là phần thay đổi quỹ đạo DO ĐỒ THỊ gây ra.")
else:
    print("Chưa ghi quỹ đạo nào (RECORD_TRAJECTORY=False).")


In [ ]:
# Vẽ lại hình đã lưu (heatmap + đường cong) cho từng tập dữ liệu và từng biến thể đồ thị
import matplotlib.image as mpimg

for name, run in result["runs"].items():
    pdir = Path(run["save_dir"]) / "trajectories"
    if not pdir.exists():
        continue
    for kind in sorted({p.stem.split("_")[-1] for p in pdir.glob("heatmap_*.png")}):
        png = pdir / f"heatmap_{kind}.png"
        if png.exists():
            print(f"[{name}] {png.name}")
            plt.figure(figsize=(9, 4))
            plt.imshow(mpimg.imread(png))
            plt.axis("off")
            plt.tight_layout()
            plt.show()

In [ ]:
# Một người học cụ thể: in bảng dài m(u,c,t) quanh các bước chuyển đổi quan trọng
name0 = list(result["runs"])[0]
run0 = result["runs"][name0]
mfile = Path(run0["save_dir"]) / "trajectories" / "mastery_train.csv"
if mfile.exists():
    mdf = pd.read_csv(mfile)
    print(mdf.head(12).to_string(index=False))
    piv = (mdf[mdf["answered"] == 1]
           .assign(d=lambda d: d["m_strong"] - d["m_pre"])
           .groupby("correct")["d"].agg(["mean", "count", "std"]))
    print("\nΔm tại khái niệm được trả lời, nhóm theo nhãn (phải dương khi đúng, âm khi sai):")
    display(piv)

## 6. Hiệu chỉnh dự đoán: ECE + Brier *(tuỳ chọn — mặc định bỏ qua vì `RUN_CALIBRATION=False`)*

Đề bài Cấp độ 2 chỉ yêu cầu **một** phân tích nâng cao và đã chọn quỹ đạo mastery (§5), nên phần này tắt mặc định. Đặt `RUN_CALIBRATION = True` để chạy khi session còn thời lượng.

Cùng một định nghĩa cho mọi mô hình (`level2/calibration.py`): `Brier = mean((p−y)²)`, `ECE = Σ_b (n_b/N)|avg(p_b) − avg(y_b)|` với 10 ngăn đều nhau. Dự đoán lấy trên toàn bộ vị trí có nhãn của tập test, dùng đúng mặt nạ mà AUC/ACC đã dùng.

Vì sao cần: AUC chỉ đo **thứ tự**, không đo độ tin cậy của con số xác suất — mà con số đó chính là thứ hiển thị cho người học. Một mô hình AUC cao hơn nhưng ECE cao hơn chưa chắc đã đáng tin hơn trong hệ thống thật.

In [ ]:
for name, run in result["runs"].items():
    cal = run.get("calibration")
    if cal is None or len(cal) == 0:
        print(f"[{name}] bỏ qua (RUN_CALIBRATION=False).")
        continue
    print("=" * 72)
    print(f"[{name}] BẢNG HIỆU CHỈNH (sắp theo ECE tăng dần)")
    print("=" * 72)
    display(cal)
    png = Path(run["save_dir"]) / "reliability.png"
    if png.exists():
        import matplotlib.image as mpimg
        plt.figure(figsize=(7, 5.6))
        plt.imshow(mpimg.imread(png))
        plt.axis("off")
        plt.tight_layout()
        plt.show()

## 7. Đánh giá đa seed *(tuỳ chọn — mặc định bỏ qua vì `RUN_MULTI_SEED=False`)*

Yêu cầu "trung bình + độ lệch chuẩn của ≥ 3 seed" đã được thoả ở Cấp độ 1 cho DKT/GKT/GKT-Zero **trên đúng split này** (số liệu dẫn lại trong `README.md`). Vì GRKT nặng hơn nhiều lần và mỗi seed là một lần train riêng, phần này tắt mặc định; đặt `RUN_MULTI_SEED = True` nếu muốn thêm phương sai riêng của GRKT.

Khi chạy, GRKT được lặp lại với `seeds = [42, 123, 456]` trên **cùng một split** (seed split luôn là 42): chỉ đổi khởi tạo trọng số và thứ tự batch. Nếu đổi cả split thì phương sai báo cáo sẽ trộn hai nguồn khác nhau, không còn là kiểm chứng ổn định.

In [ ]:
for name, run in result["runs"].items():
    if "multiseed" not in run:
        print(f"[{name}] RUN_MULTI_SEED=False — bỏ qua.")
        continue
    print("=" * 72)
    print(f"[{name}] ĐA SEED GRKT (split cố định seed={SPLIT_SEED})")
    print("=" * 72)
    display(run["multiseed_raw"])
    display(run["multiseed"])

## 8. Sản phẩm và cách đưa vào báo cáo

In [ ]:
from level2.dataset_spec import CANONICAL_COLUMNS

print("=== DANH SÁCH SẢN PHẨM CẤP ĐỘ 2 ===")
for name, run in result["runs"].items():
    print(f"\n[{name}] thư mục: {run['save_dir']}")
    for key, path in run["artifacts"].items():
        p = Path(path)
        if p.exists():
            size = p.stat().st_size
            print(f"  {key:22s} {p.name:34s} {size/1024:8.1f} KB")
        else:
            print(f"  {key:22s} {path}  (CHƯA TỒN TẠI)")
    ckpts = sorted(Path(run['save_dir']).glob('GRKT-*.pt'))
    print("  checkpoints           :", [c.name for c in ckpts])

print("\nBảng nối hai tập dữ liệu:", Path(SAVE_DIR) / "level2_all_datasets.csv")
print("Schema CSV chuẩn hoá (một dạng duy nhất cho mọi tập dữ liệu):", CANONICAL_COLUMNS)
print("Muốn tải tất cả về máy trong một lần: chạy cell đóng gói ở mục 8.5 bên dưới.")

### 8.4 Trạng thái checkpoint

Chạy cell dưới bất kỳ lúc nào — kể cả sau khi §3 dừng vì `TrainingPaused` — để biết phiên trước train tới epoch/batch nào. `ckpt_GRKT-*.pt` là checkpoint giữa epoch (resume được); `GRKT-*.pt` là checkpoint cuối (chỉ trọng số, nạp là bỏ qua train).

In [ ]:
# ==== 8.4 Trạng thái checkpoint: dùng được cả khi phiên trước đã bị ngắt ====
import torch

rows = []
for ds_dir in sorted(p for p in SAVE_DIR.iterdir() if p.is_dir()):
    paths = (sorted(ds_dir.glob("ckpt_*.pt")) + sorted(ds_dir.glob("GRKT-*.pt"))
             + sorted(ds_dir.glob("baselines/ckpt_*.pt")) + sorted(ds_dir.glob("baselines/*.pt")))
    for c in paths:
        try:
            payload = torch.load(c, map_location="cpu", weights_only=False)
        except Exception as exc:
            rows.append((ds_dir.name, c.name, "-", "-", f"lỗi đọc: {exc}"))
            continue
        fmt = payload.get("format") if isinstance(payload, dict) else None
        if fmt in ("grkt-resume-v1", "level1-resume-v1"):
            rows.append((ds_dir.name, c.name, payload.get("epoch"), payload.get("batch_pos", 0),
                         f"best val AUC={float(payload.get('best_auc', -1)):.4f} — train tiếp được"))
        elif fmt == "grkt-final-v1":
            rows.append((ds_dir.name, c.name, "xong", "-",
                         f"{payload.get('epochs')} epoch, val AUC={float(payload.get('best_val_auc', -1)):.4f}"
                         " — có fingerprint, chỉ đánh giá"))
        elif isinstance(payload, dict):
            rows.append((ds_dir.name, c.name, "xong", "-",
                         "checkpoint cuối đời cũ: không fingerprint, không kiểm chứng được split"))
        else:
            rows.append((ds_dir.name, c.name, "-", "-", type(payload).__name__))

if not rows:
    print("Chưa có checkpoint nào trong", SAVE_DIR)
else:
    print(f"{'dataset':14s} {'file':32s} {'epoch':>6s} {'batch':>6s}  ghi chú")
    for r in rows:
        print(f"{r[0]:14s} {r[1]:32s} {str(r[2]):>6s} {str(r[3]):>6s}  {r[4]}")

print("\nMuốn resume phiên sau: cần ckpt_GRKT-*.pt (đang train dở) hoặc GRKT-*.pt (đã xong).")
print("Chạy §8.5 để đóng gói chúng vào level2_output.zip.")

## 8.5 Đóng gói toàn bộ sản phẩm để tải về

Cell này là **đóng gói thủ công**. Từ bản resume-hardening, cell thực nghiệm §3 **tự gọi cùng hàm `package_output`** khi gặp `TrainingPaused` hoặc lỗi, nên zip thường đã tồn tại ngay cả khi bạn quên chạy §8.5. Tuy vậy Kaggle **không tự lưu output**: vẫn phải tải file zip về trước khi session kết thúc — chạy lại cell này để đóng gói lại sau khi đã có thêm kết quả.

Kaggle xoá `/kaggle/working` khi session kết thúc, còn tab **Output** chỉ liệt kê từng file rời rạc. Cell dưới gộp cả cây `level2_out/` thành **một file zip** đặt ngay tại `/kaggle/working`, tải bằng nút tải xuống của tab Output (hoặc sidebar **Files**) **trước khi tắt session**.

Zip này **đồng thời là gói resume**: checkpoint `GRKT-<graph>.pt` (train xong), `ckpt_GRKT-<graph>.pt` (dừng giữa epoch), `baselines/<tên>.pt` + `<tên>.json` và `baselines/ckpt_<tên>.pt` (đường nền đã xong / đang train dở) nằm trong đó, kèm `split.json` mô tả đúng split đã dùng. Phiên sau upload chính file zip này làm private Kaggle Dataset rồi chạy §0.3 — xem quy trình đầy đủ ở §0.3 và §8.4.

Mặc định bỏ hai thứ không nên nằm trong zip:

* `junyi_canonical.csv` — CSV chuẩn hoá tái sinh được bằng `ensure_canonical()`, có thể hàng trăm MB.
* file khác (trừ checkpoint GRKT) lớn hơn `MAX_MB` — in danh sách kèm lý do để bạn tự quyết định, thay vì âm thầm cắt.

Checkpoint GRKT được giữ **bất kể** `MAX_MB` vì thiếu nó là mất luôn khả năng resume; `baselines/*.pt` cũng được giữ nhưng vẫn chịu giới hạn `MAX_MB`. Muốn zip nhẹ hơn nữa (chỉ để xem số liệu, **không resume được**), đặt `KEEP_CKPTS = False`.

In [ ]:
from level2.archive import package_output

# ==== 3 núm điều chỉnh (chỉ sửa 3 thứ này nếu zip không như ý) ====
MAX_MB = 150                      # bỏ file lớn hơn mức này (trừ checkpoint resume — xem dưới)
KEEP_CKPTS = True                 # False = bỏ cả .pt/.npz, zip chỉ còn số liệu + ảnh; KHÔNG resume được
SKIP_NAMES = {"junyi_canonical.csv", "junyi_canonical.csv.meta.json"}   # tái sinh được bằng ensure_canonical(), không cần tải

if not SAVE_DIR.exists():
    print(f"Không có thư mục output {SAVE_DIR} — hãy chạy cell thực nghiệm (§3) trước.")
else:
    # Cùng một hàm với cell §3 dùng để tự đóng gói khi TrainingPaused: zip cũ bị ghi đè.
    package_output(SAVE_DIR, ZIP_PATH, max_mb=MAX_MB, keep_ckpts=KEEP_CKPTS, skip_names=SKIP_NAMES)
    print("\n    Cách tải: tab Output của notebook Kaggle -> level2_output.zip -> biểu tượng tải xuống.")
    print("    Phải tải TRƯỚC khi session kết thúc (/kaggle/working bị xoá). Chạy lại cell này bất kỳ")
    print("    lúc nào để gói thêm sản phẩm — zip cũ bị ghi đè, không tạo file thừa.")
    print("    Phiên sau: upload zip làm private Kaggle Dataset -> Add Input -> §0.3 (tự dò cả zip")
    print("    đã giải nén), giữ RESUME_GRKT_CKPT=True rồi chạy lại §3.")

## 9. Sự cố thường gặp

| Hiện tượng | Nguyên nhân và cách xử lý |
|---|---|
| Chạy smoke mất hàng chục phút | `QUICK_RUN=True` nhưng `DATASETS` vẫn là `["assist2009"]` → đổi thành `["assist2009_smoke"]` (15 người học). |
| `ImportError: cannot import name 'list_kaggle_inputs' from 'level2.dataset_spec'` (file trên đĩa ĐÃ có hàm đó) | Kernel đang giữ module **cũ** trong `sys.modules`: cell clone chạy sau lần import đầu tiên nên Git đã thay file nhưng Python không nạp lại. Cách nhanh nhất: **Restart kernel rồi Run All**. Cell §0 từ bản này cũng tự xoá module `level1/level2/GRKT/GKT` khỏi `sys.modules` và in ra `HEAD đang dùng: <commit>` để kiểm tra notebook thật sự đang chạy code nào. |
| `ModuleNotFoundError: No module named 'level2'` (ngay tại cell import) | Repo trên GitHub chưa có code Cấp độ 2: notebook `git clone` từ `origin/main`, nên mọi sửa đổi phải được **commit và push** trước khi chạy. Kiểm tra: `!git -C /kaggle/working/Knowledge-Tracing log --oneline -1` và `!ls /kaggle/working/Knowledge-Tracing/level2` |
| `KeyError: [Log_Problem.csv] Không tìm thấy cột cho ...` khi chuẩn hoá Junyi | `JUNYI.columns` không có tên cột thật của file. Cell §2 in header của file dò được; thêm tên mới vào danh sách ứng viên trong `level2/dataset_spec.py`. Bản Kaggle 2018/19: `uuid` (người học) / `ucid` (khái niệm) / `upid` (câu hỏi) / `is_correct` / `timestamp_TW` / `total_sec_taken`. Nếu traceback trỏ vào dòng không tồn tại trong file local thì notebook đang chạy bản cũ: kiểm tra `!git -C /kaggle/working/Knowledge-Tracing log --oneline -1` phải khớp commit đã push |
| `FileNotFoundError: junyi cần file dữ liệu gốc` (không tìm thấy `junyi_canonical.csv`) | Gần nhất luôn là **chưa attach hoặc chưa Agree/Subscribe**: dataset Junyi đặt `requiresSubscription=True`, không bấm agree thì `Add input` im lặng không mount gì. Kiểm tra: `from level2.dataset_spec import list_kaggle_inputs; print(list_kaggle_inputs())` — in ra từng dataset đang có trong `/kaggle/input` kèm tên + dung lượng file CSV. Nếu liệt kê đủ `Log_Problem.csv` mà vẫn không dò thấy, tên thư mục/file không khớp mẫu dò: truyền thẳng `raw_path='/kaggle/input/<thư-mục>/Log_Problem.csv'`. Cell EDA (§2) bỏ qua Junyi và in hướng dẫn; cell thực nghiệm (§3) vẫn dừng có chủ đích vì báo cáo cần đủ 2 tập. |
| Notebook chết giữa chừng khi chuẩn hoá Junyi (`Killed`, "Session crashed", hết RAM) | `Log_Problem.csv` có hàng chục triệu dòng. `_canonicalize_junyi` chỉ đọc các cột khớp `JUNYI.columns` (log in ra "đọc N/M cột"), quét THEO MẢNH và mã hoá id thành số nguyên ngay trong lúc quét. RAM giữ lại chỉ là cột số nguyên đã mã hoá, nhưng `MAX_USERS` vẫn lấy mẫu SAU khi quét hết file nên **không** giảm RAM/thời gian lúc đọc, chỉ giảm phần train. Nếu vẫn hết RAM: rút `JUNYI.columns` xuống đúng tên cột thật của file, hoặc chuẩn hoá ở notebook CPU riêng rồi attach CSV kết quả. |
| Đổi `MAX_USERS`/`min_interactions`/`JUNYI.top_concepts` mà kết quả không đổi | CSV chuẩn hoá đã tồn tại nên được dùng lại nguyên trạng (log in `dùng lại CSV chuẩn hoá`; cảnh báo ở B1 nói rõ). Xoá `level2_out/junyi/junyi_canonical.csv` rồi chạy lại từ cell §2 |
| `json.decoder.JSONDecodeError` khi mở `outputs/level1/split.json` | File JSON trong repo là **con trỏ Git LFS** (xem `.gitattributes`), không phải nội dung. Code Cấp độ 2 cố tình không đọc nó: split được tái sinh bằng `load_kt_data(seed=42, train_ratio=0.6, val_ratio=0.2, min_seq_len=3, max_seq_len=200)` rồi đối chiếu từng người học bằng `_assert_align` |
| `AttributeError: 'Series' object has no attribute 'barh'` (hoặc `.hist`) khi vẽ EDA | Image Kaggle hiện chạy **pandas 3.x**, đã bỏ các accessor vẽ hình nối tiếp trên Series (`Series.barh`, `Series.hist`, `DataFrame.plot` dựng qua matplotlib backend cũ). Code trong repo đã đổi sang gọi matplotlib thuần (`axes[0].hist(...)`, `axes[1].barh(...)`); nếu gặp ở chỗ khác, sửa tương tự thay vì hạ cấp pandas. |
| GRKT AUC ≈ 0.5 vài epoch đầu | Bình thường: GRKT khởi tạo ngẫu nhiên và có `gumbel_softmax` ở cả eval. Nếu epoch cuối vẫn ≈ 0.5, tăng `EPOCHS` hoặc giảm `LR` xuống 5e-4 |
| CUDA out of memory | Giảm `BATCH_SIZE` (64 → 32), `K_HIDDEN` (16 → 8), rồi mới tới `MAX_SEQ_LEN` (200 → 120; nhớ rằng khi đó mục 4 không còn đối chiếu khớp chính xác với Cấp độ 1). `RECORD_TRAJECTORY` ghi cả vector mastery cho mọi khái niệm: giảm `trajectory_users` |
| `KeyError: ... Split của Cấp độ 1 và dữ liệu đã nạp không cùng một lượt tiền xử lý` | `MAX_USERS`/`min_interactions`/`MAX_SEQ_LEN` đã đổi sau khi sinh CSV chuẩn hoá. Xoá `level2_out/junyi/junyi_canonical.csv` rồi chạy lại từ cell chuẩn hoá |
| `AssertionError: Chuỗi khái niệm của user ... khác Cấp độ 1` | Đây là **lỗi chủ đích**: nó chứng minh Cấp độ 2 đang nhìn dữ liệu khác Cấp độ 1. Dừng và kiểm tra lại `min_seq_len`/file CSV, không được tắt assert |
| `TypeError: only length-1 arrays can be converted to Python scalars` ở `[C2-B7]` (vẽ `reliability.png`) | Đã sửa trong `level2/calibration.py`: điểm của từng ngăn vẽ bằng `ax.scatter(..., s=...)` vì `markersize` của `plot` chỉ nhận vô hướng. Nếu gặp lỗi vẽ hình tương tự sau này, các bước vẽ đã được bọc nên **số liệu CSV vẫn còn**; đặt `RESUME_GRKT_CKPT = True` rồi chạy lại cell thực nghiệm (§3) để đi tiếp từ checkpoint GRKT đã train, không tốn GPU train lại |
| Thiếu file trong `level2_output.zip` | Cell §8.5 in ra danh sách file bị bỏ kèm lý do: `SKIP_NAMES` (CSV chuẩn hoá tái sinh được) hoặc vượt `MAX_MB`. Tăng `MAX_MB` rồi chạy lại cell — zip được ghi đè. Nếu zip rỗng, cell đã chạy trước cell thực nghiệm. |
| Thời gian chạy vượt giới hạn Kaggle (9 giờ/tuần cho GPU) | Cấu hình mặc định đã tắt ablation/calibration/đa seed, nên mỗi tập chỉ còn MỘT lần train GRKT. Nếu vẫn vượt: chạy riêng từng tập (`DATASETS = ["junyi"]`, rồi `DATASETS = ["assist2009"]` ở session sau), giảm `EPOCHS` 5 → 3, hoặc chạy trước `QUICK_RUN=True` để chắc mã chạy hết chuỗi rồi mới chạy số liệu báo cáo. `RUN_BASELINES` gần như miễn phí vì nạp checkpoint Cấp độ 1 đã commit |
| `level2.grkt_api.TrainingPaused: ... hết ngân sách thời gian` | Chủ đích: `TIME_BUDGET_MIN` (mặc định 600 phút) đã hết. Checkpoint đã ghi **trước khi** raise: `ckpt_GRKT-<graph>.pt` (giữa epoch), `GRKT-<graph>.pt` (nếu biến thể đó xong), hoặc `baselines/ckpt_<tên>.pt` (đường nền đang train lại), nên không mất phần đã train. Cell §3 đã **tự đóng gói `level2_output.zip`** khi bắt gặp `TrainingPaused`, nên chỉ cần tải zip về trước khi session kết thúc; phiên sau upload + §0.3 rồi chạy lại. §8.5 chỉ là đóng gói lại thủ công. Muốn chạy dài hơn trong một phiên: tăng `TIME_BUDGET_MIN` (nhưng notebook đã tự chặn ngân sách ở 12 giờ − 45 phút để còn thời gian ghi checkpoint + tải zip). `0` = không giới hạn. |
| Resume báo lỗi `Checkpoint được tạo với cấu hình khác (...)` | Fingerprint trong checkpoint khác cấu hình hiện tại (split, `EPOCHS`... không tính, nhưng `BATCH_SIZE`/`D_HIDDEN`/`K_HIDDEN`/`K_HOP`/`ALPHA`/`SPLIT_SEED`/`MAX_SEQ_LEN`/`MAX_USERS`/`TOP_CONCEPTS` có). Đây là bảo vệ chống trộn hai lần train. Sửa cấu hình về đúng như phiên trước, hoặc xoá `ckpt_GRKT-*.pt` + `GRKT-*.pt` của tập đó để train lại từ đầu. |
| Resume nhưng mất đúng phần đang train dở | Phiên trước chạy code cũ (trước khi có checkpoint giữa epoch) nên chỉ có `GRKT-<graph>.pt`/`baselines/<tên>.pt` khi mô hình đã train xong; dừng giữa chừng thì không còn gì để nạp. Kiểm tra bằng §8.4: phải có `ckpt_GRKT-*.pt` hoặc `baselines/ckpt_*.pt` mới resume được phần dở. |
| Phiên sau vẫn chuẩn hoá lại Junyi từ đầu (15–30 phút) | Zip mặc định bỏ `junyi_canonical.csv` (hàng trăm MB). Chấp nhận được với 12 giờ/phiên **nếu vẫn attach dataset Junyi gốc** (cần để tái sinh CSV); nếu muốn mang theo, bỏ `junyi_canonical.csv` khỏi `SKIP_NAMES` trước khi zip (file ~1 GB, upload/download chậm hơn 15–30 phút tái tạo). |
| `GRKT-<graph>.pt` hỏng / nạp lỗi (`PytorchStreamReader`, `unexpected end of file`) | Mọi lần ghi checkpoint đều **nguyên tử** (`.tmp` + `os.replace`), nên session bị kill giữa lúc ghi thì file cũ vẫn nguyên. Nếu `GRKT-<graph>.pt` vẫn hỏng, `run_grkt_experiment` tự lùi về `ckpt_GRKT-<graph>.pt`; muốn nạp lại từ đó thì **xoá file hỏng** rồi chạy lại. Cả hai đều hỏng thì xoá cả hai và train lại biến thể đó. |
| Attach zip lên Kaggle nhưng `RESUME_ZIP` không thấy file zip | Kaggle **tự giải nén** dataset `.zip` khi mount, nên `/kaggle/input/<dataset>/` có thể chỉ có cây `level2_out/...` chứ không còn file zip. Cell §0.3 chấp nhận cả hai: trỏ `RESUME_ZIP` vào file zip **hoặc** thẳng thư mục `level2_out` đã giải nén. |
| Log §0.2 báo `ngân sách bị chặn theo đồng hồ phiên` | `TIME_BUDGET_MIN` lớn hơn thời gian còn lại của session (12 giờ − 45 phút dự phòng, đọc từ `/proc/1/stat`). Đây là chủ đích để checkpoint + zip kịp được ghi trước khi Kaggle cắt session; muốn chạy lâu hơn thì chia sang phiên sau thay vì nới hằng số. |
| Cảnh báo `chỉ chứa trọng số (không có fingerprint)` | Checkpoint do code cũ (trước bản resume-hardening) sinh ra nên **không kiểm chứng được** split/đồ thị/siêu tham số; vẫn nạp nhưng có thể trộn hai cấu hình. Nếu không chắc chắn, xoá `ckpt_GRKT-*.pt` / `GRKT-*.pt` / `baselines/*.pt` của tập đó rồi train lại. |
